# 01 — Data Preprocessing and Annotation Design

<details open>
<summary><strong>From EDA to Preprocessing</strong></summary>

The previous notebook showed that our inputs are short, noisy Spanish/Catalan clinical literals with abbreviations, accents, digits, punctuation, uppercase forms, and many ambiguous duplicate literals. That means preprocessing is not just a cosmetic step. It is a modeling decision.

For this project, the final neural pipeline is based on a Spanish biomedical/clinical RoBERTa backbone. Because the tokenizer was pretrained on clinical Spanish text, aggressive normalization may remove useful information before the model has a chance to represent it.
</details>

## <details open><summary>1. Why Biomedical Spanish Preprocessing Is Dangerous</summary></details>

In Basic Text Processing, we learn common operations: lowercasing, regex cleanup, punctuation removal, accent normalization, and tokenization. In Fundamentals of Machine Learning, we also learn that feature engineering shapes what the model can learn. In Neural Networks and Deep Learning, the same idea appears through representation quality: if we delete a signal before the representation layer, the network cannot recover it.

For clinical literals, the risky signals include:

- **Punctuation:** `VHC/VHB`, `radio-cúbito`, `(izq.)` may encode relations or anatomy.
- **Accents:** Spanish and Catalan forms may differ only by diacritics.
- **Case:** all-caps strings often mark abbreviations, e.g. `HTA`, `VHC`.
- **Digits:** gestational weeks, measurements, and procedure fragments can contain numbers.
- **Abbreviations:** short forms are not noise; they are clinical language.

In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    clean_required,
    clean_lowercase,
    clean_remove_accents,
    clean_remove_punctuation,
    extract_text_pattern_features,
    compare_preprocessing_effects,
)

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
ABLATION = PROJECT_ROOT / 'data' / 'interim' / 'preprocessing_ablation'

## <details open><summary>2. Required Preprocessing</summary></details>

The required final preprocessing is intentionally minimal:

```python
text = str(text)
text = text.strip()
text = re.sub(r"\s+", " ", text)
```

Our implementation also handles null-like values safely by converting them to the empty string. It preserves case, accents, punctuation, digits, and abbreviations.

In [ ]:
examples = [
    '  VHC/VHB  ',
    'miocardiopatía   dilatada',
    'HTA\tirc\n6',
    'fractura radio-cúbito (izq.)',
    'SG:40+2',
    None,
]

pd.DataFrame({
    'original': examples,
    'required_clean': [clean_required(x) for x in examples],
    'lowercase_ablation': [clean_lowercase(x) for x in examples],
    'remove_accents_ablation': [clean_remove_accents(x) for x in examples],
    'remove_punctuation_ablation': [clean_remove_punctuation(x) for x in examples],
})

**Interpretation.** The required clean only fixes spacing. The ablation columns show what would be lost if we lowercased, stripped accents, or removed punctuation. Those transformations may be useful for classical baselines, but they are not the final RoBERTa pipeline.

## <details open><summary>3. Text Pattern Features</summary></details>

We keep pattern features for analysis. They are not the final model input by themselves, but they help us reason about what preprocessing might destroy.

In [ ]:
pd.DataFrame([extract_text_pattern_features(x) for x in examples])

## <details open><summary>4. Run the Preprocessing Command</summary></details>

The command below creates the required-clean processed files and a small ablation folder. It does not train a model.

In [ ]:
subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts' / 'run_preprocessing.py')], check=True)

In [ ]:
pd.read_csv(ABLATION / 'preprocessing_ablation_summary.csv')

In [ ]:
pd.read_csv(ABLATION / 'preprocessing_ablation_examples.csv').head(12)

## <details open><summary>5. Tokenization Before Training</summary></details>

After deciding the text-cleaning policy, we still need to understand the tokenizer. RoBERTa is a Transformer-based pretrained language model, so it does not consume raw whitespace words. It consumes subword token ids produced by its tokenizer.

This connects directly to the Transformer/PLM material from the NLP course: the input representation determines the sequence length, the attention computation, and what the model can attend to. `max_length` is therefore not arbitrary. If it is too small, we truncate clinical information; if it is too large, we waste compute and memory.

This task is easier than long EMR ICD coding in one important way: our inputs are short literals rather than full discharge summaries. But it is also harder in another way: a short literal can be ambiguous because it lacks the surrounding clinical context that would normally disambiguate the ICD category.

In [ ]:
subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts' / 'analyze_tokenization.py')], check=True)

In [ ]:
token_summary = pd.read_csv(PROJECT_ROOT / 'reports' / 'tables' / 'token_length_summary.csv')
token_summary

In [ ]:
truncation = pd.read_csv(PROJECT_ROOT / 'reports' / 'tables' / 'truncation_by_max_length.csv')
truncation

In [ ]:
from IPython.display import Image, display
display(Image(filename=PROJECT_ROOT / 'reports' / 'figures' / 'fig_07_token_length_distribution.png'))
display(Image(filename=PROJECT_ROOT / 'reports' / 'figures' / 'fig_08_truncation_rate_by_max_length.png'))

**Interpretation.** With `PlanTL-GOB-ES/roberta-base-biomedical-clinical-es`, the literals are extremely short after subword tokenization too. The training p99 is 12 tokens and the maximum is 24 tokens, including special tokens. The leaderboard is similar, with p99 12 and maximum 20. None of the examples are truncated at `max_length=32`, so `32` is an evidence-based default starting point for RoBERTa experiments.

## <details open><summary>6. Final Decision</summary></details>

Final preprocessing for RoBERTa: **required light preprocessing only**. This is why we moved from EDA to preprocessing cautiously: we clean spacing, but we do not erase case, accents, punctuation, digits, or abbreviations before the biomedical Spanish tokenizer sees them.

Final tokenizer configuration starting point: **`max_length=32`** with `PlanTL-GOB-ES/roberta-base-biomedical-clinical-es`.

We keep lowercasing, accent removal, and punctuation removal as possible ablations for classical baselines because TF-IDF models may benefit from stronger normalization. But we do not replace the final RoBERTa preprocessing unless experiments later provide evidence.

This design follows a simple principle: we can always test whether normalization helps, but we should not delete biomedical Spanish information before a pretrained clinical tokenizer sees it. The next step is therefore to build baselines on the required-clean text and use ablations only as evidence, not as assumptions.